# Comparaison des facteurs originaux et de leurs variantes — STOXX EUROPE 600

Ce notebook sert à isoler l'effet de la composition du facteur. Il relance uniquement les facteurs de référence et les variantes de l'ancien Quality Score, puis recharge les quatre composites déjà calculés dans factor_family_pipeline_STOXX600.ipynb.

Les paramètres de calcul sont volontairement identiques au notebook récent : même date de départ, même percentile, même fréquence et même méthode de remplissage. Ainsi, la comparaison attribue la différence principalement aux variables et aux poids. Le facteur Quality Score Final de l'ancien notebook est reproduit explicitement à partir de sa configuration photographiée.

Le notebook produit une table complète par période, une table totale, une table de deltas par rapport au facteur original et une figure Plotly interactive par famille. Aucun calcul incrémental n'est exécuté.

In [ ]:
from pathlib import Path
import json
import importlib
import sys

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "func.py").exists():
    raise RuntimeError(
        "Le répertoire de travail Jupyter doit être la racine du projet."
    )
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import func
importlib.reload(func)

from factor_config import LOWER_IS_BETTER, signal_options
from func import (
    calculate_benchmark_performance,
    calculate_performance_ratios,
    export_backtest_results,
    load_backtest_data,
    plot_performance_comparison,
    test_composite_signals,
)

MARKET = "STOXX EUROPE 600"
BENCHMARK = "STOXX EUROPE 600"

# Ces paramètres sont alignés sur le notebook récent afin d'isoler la composition.
START_DATE = "2007-12-01"
PERCENTILE = 0.13
FILL_METHOD = "copy"
N_JOBS = 1
PERIOD_BREAKPOINTS = [2009, 2013, 2017, 2020, 2022, 2024, 2026]

NEW_VARIANTS_DIR = (
    REPO_ROOT / "exports" / "factor_family_pipeline_STOXX600_variants"
)
OUTPUT_ROOT = REPO_ROOT / "exports"
OUTPUT_NAME = "factor_original_variants_STOXX600"

BASELINE_CANDIDATES = {
    "growth": ("GROWTH_SCORE_FS_SECTOR", "Growth Avg Percentile"),
    "quality": ("Quality Avg Percentile", "MARGIN_SCORE_FS_SECTOR"),
    "momentum": ("MOMENTUM_SCORE_FS_SECTOR", "Mom Avg Percentile"),
    "value": ("VALUE_SCORE_FS_SECTOR", "Value Avg Percentile"),
    "dividend": ("Dividend Avg Percentile", "Dividend_NTM Avg Percentile"),
}


def _clone_config(config):
    """Copie une configuration de signaux sans partager ses dictionnaires internes."""
    return {
        variable: dict(options)
        for variable, options in config.items()
    }


def _old_quality_config():
    """Reproduit la configuration Quality Score Final visible dans l'ancien notebook."""
    return {
        "Quality Avg Percentile": signal_options(
            level=1.0,
            pct_1=1.0,
            higher_is_better=True,
        ),
        "Net Debt to Tot Equity": signal_options(
            diff_1=1.0,
            higher_is_better=False,
        ),
        "FCF Conversion": signal_options(
            level=1.0,
            diff_1=2.0,
            higher_is_better=True,
        ),
    }


def _quality_variant_configs():
    """Construit les ablations et les ajouts à poids nouveau représentant 25 %."""
    base = _old_quality_config()

    without_fcf_diff = _clone_config(base)
    without_fcf_diff["FCF Conversion"]["weight_diff_1"] = 0.0

    without_fcf = _clone_config(base)
    without_fcf.pop("FCF Conversion")

    plus_roe = _clone_config(base)
    plus_roe["PCT ROE"] = signal_options(
        diff_3=2.0,
        higher_is_better=True,
    )

    plus_margin = _clone_config(base)
    plus_margin["Cont Op Earning Margin"] = signal_options(
        diff_3=2.0,
        higher_is_better=True,
    )

    plus_both = _clone_config(base)
    plus_both["PCT ROE"] = signal_options(
        diff_3=1.0,
        higher_is_better=True,
    )
    plus_both["Cont Op Earning Margin"] = signal_options(
        diff_3=1.0,
        higher_is_better=True,
    )

    return {
        "quality_original_score_final": base,
        "quality_original_without_fcf_diff": without_fcf_diff,
        "quality_original_without_fcf": without_fcf,
        "quality_original_plus_roe25": plus_roe,
        "quality_original_plus_margin25": plus_margin,
        "quality_original_plus_roe_margin25": plus_both,
    }


QUALITY_VARIANT_CONFIGS = _quality_variant_configs()


def _make_baseline_config(variable):
    """Construit le facteur original présent dans le screen pour une famille."""
    return {
        variable: signal_options(
            level=1.0,
            higher_is_better=True,
        )
    }


def _validate_directions(configs):
    """Vérifie les directions brutes avant tout lancement de backtest."""
    errors = []
    for config_name, config in configs.items():
        for variable, options in config.items():
            expected = variable not in LOWER_IS_BETTER
            actual = bool(options["higher_is_better"])
            if actual != expected:
                errors.append(
                    f"{config_name}: {variable} doit être "
                    f"{'higher' if expected else 'lower'}-is-better"
                )
    if errors:
        raise ValueError("Directions invalides : " + "; ".join(errors))


print("Fonctions et configuration de comparaison chargées.")

In [ ]:
DATA_DIR = REPO_ROOT / "data"
SCREEN_PATH = DATA_DIR / "screen_aggregateCIQ.parquet"
RETURNS_PATH = DATA_DIR / "returns.parquet"

available_columns = set(pq.ParquetFile(SCREEN_PATH).schema_arrow.names)
BASELINE_COLUMNS = {}
for family, candidates in BASELINE_CANDIDATES.items():
    selected = next(
        (candidate for candidate in candidates if candidate in available_columns),
        None,
    )
    if selected is None:
        raise KeyError(
            f"Aucun facteur original disponible pour {family}: {candidates}"
        )
    BASELINE_COLUMNS[family] = selected

ORIGINAL_CONFIGS = {
    f"original_screen__{family}": _make_baseline_config(variable)
    for family, variable in BASELINE_COLUMNS.items()
}
ORIGINAL_CONFIGS.update(QUALITY_VARIANT_CONFIGS)
_validate_directions(ORIGINAL_CONFIGS)

RAW_VARIABLES = sorted(
    {
        variable
        for config in ORIGINAL_CONFIGS.values()
        for variable in config
    }
)
LOAD_VARIABLES = list(dict.fromkeys(
    RAW_VARIABLES + list(BASELINE_COLUMNS.values())
))

screen, returns = load_backtest_data(
    screen_path=SCREEN_PATH,
    returns_path=RETURNS_PATH,
    variables=LOAD_VARIABLES,
    bench=BENCHMARK,
    start_date=START_DATE,
    lookback_periods=12,
    compact_dtypes=True,
)
screen["Date"] = pd.to_datetime(screen["Date"])

missing = [
    column for column in LOAD_VARIABLES
    if column not in screen.columns
]
if missing:
    raise KeyError(f"Variables absentes après chargement : {missing}")
if f"Weight in {BENCHMARK}" not in screen.columns:
    raise KeyError(
        f"La colonne Weight in {BENCHMARK} est absente du screen."
    )

bench_perf = calculate_benchmark_performance(
    screen=screen,
    returns=returns,
    bench=BENCHMARK,
    start_date=START_DATE,
)

RUN_OPTIONS = {
    "bench": BENCHMARK,
    "bench_perf": bench_perf,
    "percentile": PERCENTILE,
    "start_date": START_DATE,
    "freq_rebal": 1,
    "fill_method": FILL_METHOD,
    "n_jobs": N_JOBS,
    "retain_builders": False,
    "monthly_base_cache": {},
    "period_breakpoints": PERIOD_BREAKPOINTS,
    "show_plot": False,
    "build_figure": False,
}

print(f"screen={screen.shape}, returns={returns.shape}")
print(f"Facteurs originaux détectés : {BASELINE_COLUMNS}")
print(f"Variantes Quality à tester : {list(QUALITY_VARIANT_CONFIGS)}")

In [ ]:
original_batch = test_composite_signals(
    screen=screen,
    returns=returns,
    composite_configs=ORIGINAL_CONFIGS,
    list_noire_path=None,
    score_prefix="Score_OriginalVariant",
    **RUN_OPTIONS,
)

exported = export_backtest_results(
    results={"original_and_variants": original_batch},
    output_dir=OUTPUT_ROOT,
    export_name=OUTPUT_NAME,
    export_html=False,
    export_png=False,
    export_holdings=False,
)
ORIGINAL_DIR = Path(exported["export_dir"])

print(f"Backtests originaux et variantes terminés : {ORIGINAL_DIR}")
print("Les nouveaux composites ne sont pas recalculés dans ce notebook.")

In [ ]:
if not NEW_VARIANTS_DIR.exists():
    raise FileNotFoundError(
        "Le dossier des nouveaux composites est absent : "
        f"{NEW_VARIANTS_DIR}. "
        "Modifiez NEW_VARIANTS_DIR vers le dossier produit par "
        "factor_family_pipeline_STOXX600.ipynb."
    )

def _read_export(export_dir):
    """Recharge les metrics et les performances enregistrées d'une expérience."""
    metrics_path = export_dir / "backtest_metrics.csv"
    if not metrics_path.exists():
        raise FileNotFoundError(f"backtest_metrics.csv absent : {metrics_path}")
    return (
        pd.read_csv(metrics_path),
        func._load_saved_performances(export_dir),
    )


original_metrics, original_sources = _read_export(ORIGINAL_DIR)
new_metrics, new_sources = _read_export(NEW_VARIANTS_DIR)

def _find_rows(metrics, target):
    """Retrouve un test par son nom exact ou son suffixe dans le registre."""
    target = str(target)
    test_name = metrics["test_name"].astype(str)
    test_path = metrics["test_path"].astype(str)
    mask = (
        test_name.eq(target)
        | test_path.eq(target)
        | test_name.str.endswith(target)
        | test_path.str.endswith(target)
    )
    return metrics.loc[mask].copy()


def _find_source(metrics, sources, target):
    """Retrouve la performance liée à un test enregistré."""
    rows = _find_rows(metrics, target)
    if rows.empty:
        raise KeyError(f"Test absent de l'export : {target}")
    test_path = str(rows.iloc[0]["test_path"])
    if test_path in sources:
        return rows, sources[test_path]
    for source_path, source in sources.items():
        if str(source.get("test_name")) == target:
            return rows, source
    raise KeyError(f"Performance absente du registre : {target}")


METRIC_COLUMNS = [
    "active_cagr",
    "top_worst_cagr",
    "top_information_ratio",
    "robust_score",
    "active_max_drawdown",
    "tracking_error_annualized",
    "min_rolling_3y_cagr",
    "top_bench_ratio",
    "top_worst_ratio",
    "top_annualized_return",
    "bench_annualized_return",
]

comparison_specs = []
for family in BASELINE_COLUMNS:
    comparison_specs.append({
        "family": family,
        "label": f"original_screen__{family}",
        "target": f"original_screen__{family}",
        "source_group": "original",
        "metrics": original_metrics,
        "sources": original_sources,
        "directory": ORIGINAL_DIR,
    })
    for variant_name in (
        "stable_core",
        "recent_confirmation",
        "current_control",
        "pooled_long_horizon",
    ):
        comparison_specs.append({
            "family": family,
            "label": f"new__{variant_name}",
            "target": f"{variant_name}__{family}",
            "source_group": "new_composite",
            "metrics": new_metrics,
            "sources": new_sources,
            "directory": NEW_VARIANTS_DIR,
        })

for target in QUALITY_VARIANT_CONFIGS:
    comparison_specs.append({
        "family": "quality",
        "label": target,
        "target": target,
        "source_group": "old_quality_variant",
        "metrics": original_metrics,
        "sources": original_sources,
        "directory": ORIGINAL_DIR,
    })

metric_parts = []
for specification in comparison_specs:
    rows = _find_rows(
        specification["metrics"],
        specification["target"],
    )
    if rows.empty:
        raise KeyError(
            f"Le test {specification['target']} est absent de "
            f"{specification['directory']}"
        )
    rows["family"] = specification["family"]
    rows["comparison_label"] = specification["label"]
    rows["source_group"] = specification["source_group"]
    rows["source_directory"] = str(specification["directory"])
    metric_parts.append(rows)

combined_metrics = pd.concat(metric_parts, ignore_index=True)
for column in METRIC_COLUMNS:
    combined_metrics[column] = pd.to_numeric(
        combined_metrics[column],
        errors="coerce",
    )

combined_metrics.to_csv(
    ORIGINAL_DIR / "comparison_original_variants_all.csv",
    index=False,
)
combined_metrics.loc[
    combined_metrics["period_id"].astype(str).eq("total")
].to_csv(
    ORIGINAL_DIR / "comparison_original_variants_total.csv",
    index=False,
)

base_label_by_family = {
    family: (
        "quality_original_score_final"
        if family == "quality"
        else f"original_screen__{family}"
    )
    for family in BASELINE_COLUMNS
}
base_rows = combined_metrics.loc[
    combined_metrics["comparison_label"].isin(base_label_by_family.values()),
    ["family", "comparison_label", "period_id", "scope", *METRIC_COLUMNS],
].copy()
base_rows["base_label"] = base_rows["family"].map(base_label_by_family)
base_rows = base_rows.loc[
    base_rows["comparison_label"].eq(base_rows["base_label"])
].drop(columns=["base_label", "comparison_label"])
base_rows = base_rows.rename(
    columns={column: f"{column}_original" for column in METRIC_COLUMNS}
)

comparison = combined_metrics.merge(
    base_rows,
    on=["family", "period_id", "scope"],
    how="left",
)
for column in METRIC_COLUMNS:
    comparison[f"delta_{column}"] = (
        comparison[column] - comparison[f"{column}_original"]
    )
comparison["performance_gate"] = (
    comparison["active_cagr"].gt(0)
    & comparison["top_worst_cagr"].gt(0)
    & comparison["top_information_ratio"].gt(0)
)
comparison["improves_three_metrics"] = (
    comparison["delta_active_cagr"].gt(0)
    & comparison["delta_top_worst_cagr"].gt(0)
    & comparison["delta_top_information_ratio"].gt(0)
)
comparison["risk_not_worse"] = (
    comparison["delta_active_max_drawdown"].le(0)
    & comparison["delta_tracking_error_annualized"].le(0)
)
comparison.to_csv(
    ORIGINAL_DIR / "comparison_vs_original.csv",
    index=False,
)
comparison.loc[
    comparison["period_id"].astype(str).eq("total")
].to_csv(
    ORIGINAL_DIR / "comparison_vs_original_total.csv",
    index=False,
)

contract_columns = [
    "benchmark",
    "percentile",
    "start_date",
    "fill_method",
    "frequency_rebalancing",
]
contract_rows = []
for directory, source_group, metrics in (
    (ORIGINAL_DIR, "original_and_variants", original_metrics),
    (NEW_VARIANTS_DIR, "new_composites", new_metrics),
):
    available = [
        column for column in contract_columns
        if column in metrics.columns
    ]
    if not available:
        continue
    row = {
        "source_group": source_group,
        "directory": str(directory),
    }
    for column in available:
        values = metrics[column].dropna().astype(str).unique()
        row[column] = " | ".join(values[:5])
    contract_rows.append(row)
pd.DataFrame(contract_rows).to_csv(
    ORIGINAL_DIR / "comparison_contracts.csv",
    index=False,
)

print(f"Comparaison exportée dans : {ORIGINAL_DIR}")

In [ ]:
def _performance_for(metrics, sources, target):
    """Recharge la courbe Top et la courbe Bench d'un test."""
    rows, source = _find_source(metrics, sources, target)
    performance = source["performance"].copy()
    required = {"Top", "Bench"}
    missing = required.difference(performance.columns)
    if missing:
        raise KeyError(f"Colonnes de performance absentes pour {target}: {missing}")
    return performance.sort_index()

def _family_performance(family, specifications):
    """Combine les courbes Top de plusieurs expériences et une référence Bench."""
    top_series = []
    benchmark = None
    for specification in specifications:
        performance = _performance_for(
            specification["metrics"],
            specification["sources"],
            specification["target"],
        )
        top_series.append(
            performance["Top"].rename(specification["label"])
        )
        if benchmark is None:
            benchmark = performance["Bench"].rename("Benchmark")
    combined = pd.concat([*top_series, benchmark], axis=1).sort_index()
    combined = combined.loc[combined["Benchmark"].notna()]
    ratios = calculate_performance_ratios(
        combined,
        benchmark_column="Benchmark",
    )
    return combined, ratios

figure_paths = {}
for family in BASELINE_COLUMNS:
    family_specs = [
        specification
        for specification in comparison_specs
        if specification["family"] == family
    ]
    performance, ratios = _family_performance(family, family_specs)
    performance.to_csv(
        ORIGINAL_DIR / f"performance_{family}_original_variants.csv",
        index=True,
    )
    ratios.to_csv(
        ORIGINAL_DIR / f"performance_{family}_original_variants_ratios.csv",
        index=True,
    )
    figure_path = (
        ORIGINAL_DIR / "figures" / f"comparison_original_variants_{family}.html"
    )
    figure_path.parent.mkdir(parents=True, exist_ok=True)
    figure = plot_performance_comparison(
        performance=performance,
        ratios=ratios,
        benchmark_column="Benchmark",
        title=f"STOXX EUROPE 600 | {family} | original et variantes",
        save_path=figure_path,
        show_plot=False,
        rebase=True,
        period_breakpoints=PERIOD_BREAKPOINTS,
        show_worst_performance=False,
    )
    figure_paths[family] = str(figure_path)
    display(figure)

with (ORIGINAL_DIR / "comparison_manifest.json").open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        {
            "market": MARKET,
            "benchmark": BENCHMARK,
            "start_date": START_DATE,
            "percentile": PERCENTILE,
            "fill_method": FILL_METHOD,
            "period_breakpoints": PERIOD_BREAKPOINTS,
            "new_variants_directory": str(NEW_VARIANTS_DIR),
            "original_directory": str(ORIGINAL_DIR),
            "quality_variants": list(QUALITY_VARIANT_CONFIGS),
            "figure_paths": figure_paths,
            "outputs": [
                "comparison_original_variants_all.csv",
                "comparison_original_variants_total.csv",
                "comparison_vs_original.csv",
                "comparison_vs_original_total.csv",
                "comparison_contracts.csv",
                "performance_<family>_original_variants.csv",
                "performance_<family>_original_variants_ratios.csv",
                "figures/comparison_original_variants_<family>.html",
            ],
        },
        handle,
        ensure_ascii=False,
        indent=2,
    )

print("Figures Plotly et manifest enregistrés.")

In [ ]:
DISPLAY_COLUMNS = [
    "family",
    "comparison_label",
    "period_id",
    "period_label",
    "active_cagr",
    "top_worst_cagr",
    "top_information_ratio",
    "robust_score",
    "active_max_drawdown",
    "tracking_error_annualized",
    "min_rolling_3y_cagr",
    "performance_gate",
    "improves_three_metrics",
    "risk_not_worse",
]

total_view = comparison.loc[
    comparison["period_id"].astype(str).eq("total"),
    DISPLAY_COLUMNS,
].sort_values(["family", "active_cagr"], ascending=[True, False])

quality_view = comparison.loc[
    comparison["family"].eq("quality"),
    DISPLAY_COLUMNS + [
        "delta_active_cagr",
        "delta_top_worst_cagr",
        "delta_top_information_ratio",
        "delta_robust_score",
    ],
].sort_values(["period_id", "active_cagr"], ascending=[True, False])

print("=" * 100)
print("RESULTATS TOTAUX : facteur original, variantes Quality et nouveaux composites")
display(total_view)
print("=" * 100)
print("QUALITY PAR PERIODE : comparaison détaillée et deltas")
display(quality_view)
print("=" * 100)
print(
    "Pour l'analyse suivante, envoyez comparison_vs_original_total.csv, "
    "comparison_vs_original.csv et comparison_contracts.csv."
)